# Error detection from the carrier thoughts: SIM-CoT CODI-1B (ledger §125, Build 2)

**No training** of the model. Records the six thought states, the native answer and the
forced-cue confidence for every GSM8K test question (right or wrong), then asks whether the
model's own numeric readout at the card's carrier slots predicts wrong answers better than
inert slots or confidence alone (prediction P10). No dataset to attach. Publish the output as
a dataset when finished (Output tab -> New Dataset).

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "74da87bcc553beedae0770d2539a970022e66c8a"
REPO_DIR = "/kaggle/working/latent-reasoning"
CONFIG = "configs/official_simcot_codi_llama1b.yaml"
MODEL = "simcot"
PRECISION = "float16"
OUTPUT_ROOT = "/kaggle/working/readout_simcot"
CARD = "docs/latent_monitor_audit/simcot_gsm8k_test.card.json"
INERT_OVERRIDE = None
BATCH_SIZE = "32"
RUN_SMOKE = True

import glob, json, os, pathlib, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_official_codi_family.py", "tests/test_cache_carrier.py", "tests/test_correctness_readout.py", "tests/test_monitor_faithfulness.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## Step 1: record states, native answers and confidence (smoke first, then the full test set)

In [ ]:
def record(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_codi_correctness_readout.py", "--config", CONFIG, "--output-dir", out,
           "--batch-size", BATCH_SIZE, "--precision", PRECISION, "--device", "cuda"]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "summary.json").read())

if RUN_SMOKE:
    record(OUTPUT_ROOT + "_smoke", True)
summary = record(OUTPUT_ROOT, False)
print("native accuracy", summary["native_accuracy"], "published", summary["published_accuracy"], "rows", summary["rows"])

## Step 2: the analysis, slots chosen by the card

In [ ]:
run_with_retry([sys.executable, "-u", "scripts/analyze_correctness_readout.py", "--config", CONFIG, "--card", CARD, OUTPUT_ROOT,
                "--output", f"{OUTPUT_ROOT}/analysis.json", *(["--inert", INERT_OVERRIDE] if INERT_OVERRIDE else [])])
report = json.loads(open(f"{OUTPUT_ROOT}/analysis.json").read())
from scripts.analyze_correctness_readout import markdown
from IPython.display import Markdown, display
display(Markdown(markdown(report)))
print(json.dumps(report["checks"], indent=2))
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")

## Keep the output under Kaggle's 500-file cap

In [ ]:
import shutil
shutil.rmtree(REPO_DIR, ignore_errors=True)   # the clone (with .git) is not an output; without this the run's own files can be dropped from the saved output
print("removed", REPO_DIR, "; remaining output files:", sum(len(f) for _, _, f in os.walk("/kaggle/working")))